In [1]:
# %matplotlib widget
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from helper_functions import filtering, analysis, plotting, diagnostics
from obspy import read
import traceback 
import time
import obspy
from station_keys import STATION_PHASE_MAP
import sys

In [2]:
# ==================== Load data ====================
# the next two datasets are 200 400 phase windowed datasets with newer updated sta lta values

dataType = 'complete'
networkName = 'swp'
stationName = 'swp'
freqType = 'normal'

raw_swp_eventdata = pd.read_csv(
    f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
    sep='|'
)

dataType = 'complete'
networkName = 'AU'
stationName = 'australia'
freqType = 'normal'

raw_aus_eventdata = pd.read_csv(
    f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
    sep='|'
)

raw_eventdata = pd.concat([raw_swp_eventdata, raw_aus_eventdata], ignore_index=True)
raw_eventdata = raw_eventdata.sort_values('evstation')

# ==================== Prepare data ====================
# Calculate SI errors
eventData = filtering.calculate_si_errors(raw_eventdata)

print("length before filtering:", len(eventData))
s_count = (eventData["evphase"] == 'S').sum()
sks_count = (eventData["evphase"] == 'SKS').sum()

print("S count:", s_count)
print("SKS count:", sks_count)

print("Number of Seismic Stations Before Filtering:", eventData['evstation'].nunique())
print(eventData['evstation'].unique())
# print(eventData['evstation'].value_counts())



epicentral_max=360
epicentral_min=0
depth_min=0
depth_max=1e5
snrbefore_min=3.0
snrafter_min=1.0
sta_lta_min=0
mag_max=7.0
ellipticity_min=1.0
ellipticity_max=1e4
si_error_max=0.5
incident_max=90
pierce_depth_min=0
delta_travel_max = 25


# Apply quality filters
eventData = filtering.filter_events(
    eventData,
    epicentral_max=epicentral_max,
    epicentral_min=epicentral_min,
    depth_min=depth_min,
    depth_max=depth_max,
    snrbefore_min=snrbefore_min,
    snrafter_min=snrafter_min,
    sta_lta_min=sta_lta_min,
    mag_max=mag_max,
    ellipticity_min=ellipticity_min,
    ellipticity_max=ellipticity_max,
    si_error_max=si_error_max,
    incident_max=incident_max,
    pierce_depth_min=pierce_depth_min,
    delta_travel_max=delta_travel_max
)

eventData = eventData.drop_duplicates(subset=['evtime','evstation'])

# Remove shallow events with large incident angles (keeps only events with large epicentral distances and smaller incident angles)
eventData = eventData[~((eventData['evdp'] < 400) & (eventData['evincident'] > 25))]

# # Remove deep events with very large incident angles (sample)
eventData = eventData[~((eventData['evdp'] >= 400) & (eventData['evincident'] > 35))]

# ^^^^^^ these two filters are heavily used in the new analysis so keep them!!!


'''
april 5 notes
- added tilt to SI calculation and plotting (see analysis.py and spring_updates.ipynb)
- started looking at treating deep S phases and direct S phases separately (mainly dealing with incident angle and pierce depth filters)
- analysis of only deep S phases to identify trends and how this could be corraborated with sampling of anisotropy at tilted layer)
'''
print("length after filtering:", len(eventData))
s_count = (eventData["evphase"] == 'S').sum()
sks_count = (eventData["evphase"] == 'SKS').sum()

print("S count:", s_count)
print("SKS count:", sks_count)

print("Number of Seismic Stations After Filtering:", eventData['evstation'].nunique())


print(f"Events after filtering: {len(eventData)}")
print(eventData['evstation'].unique())

print(eventData['evstation'].value_counts().to_string())


length before filtering: 99224
S count: 82161
SKS count: 17063
Number of Seismic Stations Before Filtering: 115
['ARMA' 'AS31' 'BAG' 'BBOO' 'BCON' 'BKB' 'BKNI' 'BLDU' 'BTDF' 'CAN' 'CHBT'
 'CISI' 'CMSA' 'CNB' 'COEN' 'CTA' 'CTAO' 'DAV' 'DLV' 'EIDS' 'FAKI' 'FITZ'
 'FORT' 'GIRL' 'GUMO' 'HNR' 'HTT' 'INKA' 'IPM' 'JAGI' 'JAY' 'KDU' 'KKM'
 'KLGGI' 'KLGHH' 'KMBL' 'KNRA' 'KOM' 'KOR' 'KSM' 'KUM' 'LCRK' 'LDM' 'LHI'
 'LUWI' 'MANU' 'MEEK' 'MILA' 'MMRI' 'MNAI' 'MOO' 'MORW' 'MTN' 'MULG' 'MUN'
 'NFK' 'NOUC' 'NWAO' 'OOD' 'PATS' 'PBKT' 'PMBI' 'PMG' 'PVC' 'QIS' 'QLP'
 'RABL' 'RIV' 'SAUI' 'SBM' 'SKLT' 'SMRI' 'SNZO' 'SOEI' 'STKA' 'SZP' 'TMDB'
 'TNTI' 'TOLI' 'TOO' 'TROP' 'UBIN' 'UBPT' 'UGM' 'WAKE' 'WB0' 'WB1' 'WB10'
 'WB2' 'WB3' 'WB4' 'WB5' 'WB6' 'WB7' 'WB8' 'WB9' 'WC1' 'WC2' 'WC3' 'WC4'
 'WR0' 'WR1' 'WR10' 'WR2' 'WR3' 'WR4' 'WR5' 'WR6' 'WR7' 'WR8' 'WR9' 'WRKA'
 'XMI' 'XMIS' 'YNG']
length after filtering: 8012
S count: 7089
SKS count: 923
Number of Seismic Stations After Filtering: 83
Events after filtering:

# Generate Plots for Single Defined Station

In [ ]:
print(eventData['evstation'].unique())

truncatedData = eventData.copy()
truncatedData = truncatedData[truncatedData['evstation']=='EIDS']
truncatedData = truncatedData[truncatedData['evphase']=='SKS']
# # truncatedData = truncatedData[truncatedData['evdp']>=200]
truncatedData = truncatedData[(truncatedData['evbaz']>=135) & (truncatedData['evbaz']<=150)]
# # truncatedData = truncatedData[truncatedData['wiggleSI_calc']>=0]


truncatedData = truncatedData[['evstation','evtime','evlat','evlon','evdp','evmg','evphase','evepic','evbaz','calcbaz','wiggleSI_calc']]
truncatedData = truncatedData.sort_values(by='evepic')

truncatedData


s_truncatedData = truncatedData[truncatedData['evphase']=='S']
sks_truncatedData = truncatedData[truncatedData['evphase']=='SKS']

sks_shallow_truncatedData = sks_truncatedData[sks_truncatedData['evdp']<400]
sks_deep_truncatedData = sks_truncatedData[sks_truncatedData['evdp']>=400]

# plt.plot(truncatedData['evbaz'], truncatedData['wiggleSI_calc'], 'o',c='b')
# plt.plot(truncatedData['calcbaz'], truncatedData['wiggleSI_calc'], 'o',c='r')

# plt.plot(s_truncatedData['evbaz'], s_truncatedData['wiggleSI_calc'], 'o',c='r', label='S')
# plt.plot(sks_truncatedData['evbaz'], sks_truncatedData['wiggleSI_calc'], 'o',c='b', label='SKS')

plt.figure(figsize=(8,6))
# plt.plot(s_truncatedData['evepic'], s_truncatedData['wiggleSI_calc'], 'o',c='r', label='S')
plt.plot(sks_truncatedData['evepic'], sks_truncatedData['wiggleSI_calc'], 'o',c='blue', label='SKS')

# plt.plot(sks_shallow_truncatedData['evepic'], sks_shallow_truncatedData['wiggleSI_calc'], 'o',c='blue', label='SKS shallow')
# plt.plot(sks_deep_truncatedData['evepic'], sks_deep_truncatedData['wiggleSI_calc'], 'o',c='green', label='SKS deep')
plt.xlabel('Epicentral Distance [deg]')
# plt.xlim(100,200)
plt.ylabel('SI')
# plt.yticks(np.linspace(-2,2,9))
# plt.ylim(-2,2)
plt.legend()
# plt.title(' SKS events with 120-150 baz')
plt.show()

print(f" Mean source latitude: {round(np.mean(truncatedData['evlat']),2)}")
print(f" Mean source longitude: {round(np.mean(truncatedData['evlon']),2)}")


truncatedData.to_csv('/work/gcl3/BR/nicolasv03/senior_thesis/codes/Plotting/EIDS_evbaz_135_to_150_events.txt')

# MAIN CODE FOR PUBLICATION QUALITY PLOTS

In [ ]:


# Select station
# station_events = eventData 
station_code = 'ARMA'
station_events = eventData[eventData['evstation'] == station_code]
without_cluster = True

# # SKS and deep S only filter:
# station_events = station_events[
#     (station_events['evphase'] == 'SKS') |  # Keep if SKS
#     ((station_events['evphase'] == 'S') & (station_events['evdp'] >= 400)) # OR if deep S
# ]
    

# ARMA SKS cluster removal: between 120 - 150 
# BBOO SKS cluster removal: between 145 - 165

if without_cluster:
    station_events = station_events[
        (station_events['evphase'] != 'SKS') |  # Keep if not SKS
        (station_events['calcbaz'] < 130) |      # OR if calcbaz < 145
        (station_events['calcbaz'] > 160)      # OR if calcbaz > 165
    ]

# # # # BBOO remove S above 200 baz 
# station_events = station_events[
#     (station_events['evphase'] != 'S') |  # Keep if not S
#     (station_events['calcbaz'] < 200)      # OR if calcbaz < 200
# ]

# EIDS remove super strong deep S SI values above 1.5
# MULG  remove super strong deep S SI values above 1.8
# SNZO  remove super strong deep S SI values above 2
# TOO  remove super strong deep S SI values above 1.4
# YNG remove superstrong SI values above 1.8



station_events = station_events[abs(station_events['wiggleSI_calc'])<1.8]

bin_width = 20
bin_minimum = 3
output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/polarization_azimuth_plots/{station_code}/'

phase_combo = STATION_PHASE_MAP[station_code]
# phase_combo = "sks"
# phase_combo = "sks_deep"
# phase_combo = "all"


results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False,with_C=False)


fig, ax = analysis.publication_SI_curve(results, station_code,phase_combo,s_depth=400,figsize=(12,8),with_C=False)

pol_azi_output = output_path+f'{station_code}_ylim2_polarization_azimuth_plot_{phase_combo}'
if without_cluster:
    pol_azi_output = pol_azi_output + "_without_cluster"

# plt.savefig(f"{pol_azi_output}.png", dpi=300, bbox_inches='tight')

plt.show()

fig2, ax2 = plotting.plot_station_stereographic_map(results, station_code,phase_combo, s_depth=400, figsize=(10, 10))
output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/polarization_azimuth_plots/'

pol_azi_map_output = output_path+f'raypath_maps/{station_code}_event_distribution_map_{phase_combo}'
if without_cluster:
    pol_azi_map_output = pol_azi_map_output + "_without_cluster"

plt.savefig(f"{pol_azi_map_output}.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
print(results['sks_deep']['result_binned']['cov'])
pcov = results['sks_deep']['result_binned']['cov']
perr = np.sqrt(np.diag(pcov))
print(perr)

In [ ]:
results['sks_deep']['result_binned']

# loop to get splitting parameters

In [ ]:
'''
Doing this just to get splitting parameters for the stations so we can start making a publiation quality map. Final measurements coming soon!! May 11 2026

'''

# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Filter to only stations in dictionary
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
print(f"After dict filter: {len(station_list)} stations remaining")

print(station_list)

plotting_parameters = pd.DataFrame(columns=["Station","Network","Latitude","Longitude","phi","dt","sks_count","s_deep_count","s_shallow_count","total_count"])

# Loop through stations
for station_code in station_list:
    time.sleep(1)
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    time.sleep(1)
    # if STATION_PHASE_MAP[station_code] == 'complex':
    #     print("Station is complex and will not be analyzed.")
    #     continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]

        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        
        result_binned = data['result_binned']
        station_events = data['station_events']
        
        # Extract parameters
        phi_binned = round(result_binned['phi'],1)
        dt_binned = round(result_binned['dt'],2)

        station_lat = round(station_events['evstationlat'].iloc[0],2)
        station_lon = round(station_events['evstationlon'].iloc[0],2)

        network_name = station_events['evnetwork'].iloc[0]
        print(network_name)

        sks_count = len(station_events[station_events['evphase'] == 'SKS'])
        s_deep_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] >= s_depth)])
        s_shallow_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] < s_depth)])
        total_count = sks_count + s_deep_count + s_shallow_count
        print(sks_count,s_deep_count,s_shallow_count)




        # print(phi_binned,dt_binned,station_lat,station_lon)

        plotting_parameters.loc[len(plotting_parameters)] = [station_code,network_name,station_lat,station_lon,phi_binned,dt_binned,sks_count,s_deep_count,s_shallow_count,total_count]
        # ######################################################
        # print(data['station_events'].head())


        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/plots_aug1/'

        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=True)
        fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)

        plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_plot.png', dpi=300, bbox_inches='tight')
        
        
        print(f"  ✓ Completed {station_code}")

        plt.close('all')

        results = []
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

# print(f"\n{'='*70}")
# print(f"✓ Processing complete for {len(station_list)} stations")
# print('='*70)

print(plotting_parameters)

# plotting_parameters.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/publication_map/swp_measurements_aug1_all_stations.txt",sep='\t')

# BLOB IDENTIFIER

In [ ]:
# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

# blob_stations = ['ARMA','BBOO','CAN','CNB','YNG']

blob_stations = ['ARMA'] #,'BBOO','CAN','CNB','YNG']

# 'BCON'
# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Filter to only stations in dictionary and in blob list
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
station_list = [s for s in station_list if s in blob_stations]

print(f"After dict filter: {len(station_list)} stations remaining")

model = obspy.taup.TauPyModel(model="iasp91")

from obspy.geodetics.base import locations2degrees,gps2dist_azimuth

from pyproj import Geod

g = Geod(ellps='WGS84')

blob_coords = pd.DataFrame(columns=['lat','lon','evlat','evlon','statlat','statlon','evdp','evphase','evepic','SI','evbaz'])

blob_tick = 0 

# Loop through stations
for station_code in station_list:
    
    # if STATION_PHASE_MAP[station_code] == 'complex':
    #     continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]
        # print(station_code)
        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        # print(data)
        # print(data.keys())
        # time.sleep(1)
        events = data['station_events']
        blob_events = events[(events['evphase'] == 'SKS') & 
                             (events['calcbaz'] > 130) &
                             (events['calcbaz'] < 160) 
        ]

        # print(len(blob_events))
        for i in range(0,len(blob_events)):
            event = blob_events.iloc[i]
        
            arr = model.get_pierce_points(  source_depth_in_km=event['evdp'],
                                            distance_in_degree=event['evepic'],
                                            phase_list=['SKS'])
            
            # pierce = np.array(arr[0].pierce)
            # cmb_inds = pierce[:]
            pierce_array = np.array(arr[0].pierce.tolist())
            cmb_inds = np.where(np.abs(pierce_array[:,3] - 2889) < 20)[0]
            receiver_cmb = pierce_array[cmb_inds[-1]]
            cmb_dist_deg = np.degrees(receiver_cmb[2])
            # print(receiver_cmb)
            # print(cmb_dist_deg)


            dist_m, az, baz = gps2dist_azimuth(event['evlat'],
                                                event['evlon'],
                                                event['evstationlat'],
                                                event['evstationlon'])


            total_deg = locations2degrees(
                                                event['evlat'],
                                                event['evlon'],
                                                event['evstationlat'],
                                                event['evstationlon'],
                                            )
            fraction = cmb_dist_deg / total_deg

            pierce_dist_m = dist_m * fraction

            # print(fraction)



            cmb_lon, cmb_lat, backaz = g.fwd(
                event['evlon'],
                event['evlat'],
                az,
                pierce_dist_m
            )

            if cmb_lon < 150:
                print('cmb_lon less than 150')
                continue

            if event['evlon'] < -100:
                print('evlon less than -100')
                continue

            # print(cmb_lat, cmb_lon)

            blob_coords.loc[blob_tick,:] = [cmb_lat,cmb_lon,event['evlat'],event['evlon'],event['evstationlat'],event['evstationlon'],event['evdp'],event['evphase'],event['evepic'],event['wiggleSI_calc'],event['evbaz']]

            blob_tick += 1

            

        # plot the blob
        # output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/blob_plots/'
        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)
        # plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_plot.png', dpi=300, bbox_inches='tight')
        
        # station_events = station_events[
#     (station_events['evphase'] != 'SKS') |  # Keep if not SKS
#     (station_events['calcbaz'] < 120) |      # OR if calcbaz < 145
#     (station_events['calcbaz'] > 150)      # OR if calcbaz > 165
# ]


        results = []
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        traceback.print_exc()

        plt.close('all')  # Clean up any open figures
        continue





In [ ]:
# blob_coords.to_csv

blob_coords.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/blob_plots/blob_coordinates_ARMA_only.txt",sep='\t')

In [ ]:
data['station_events']

In [ ]:
print(np.mean(blob_coords['statlat']))
print(np.mean(blob_coords['statlon']))

In [ ]:
blob_coords.sort_values()

blob_coords.head(len(blob_coords)-59)

# CLUSTER REMOVAL MEASUREMENT CORRECTION

we got our first batch of measurements, lets finally remove the cluster and see how our results change!

In [ ]:
'''
Doing this just to get splitting parameters for the stations so we can start making a publiation quality map. Final measurements coming soon!! May 11 2026

'''

# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

blob_stations = ['ARMA','BBOO','CAN','CNB','YNG']

# Filter to only stations in dictionary and in blob list
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
station_list = [s for s in station_list if s in blob_stations]

print(f"After dict filter: {len(station_list)} stations remaining")

print(station_list)

# here we remove the sks events...
     # events = data['station_events']
        #         blob_events = events[(events['evphase'] == 'SKS') & 
        #                              (events['calcbaz'] > 130) &
        #                              (events['calcbaz'] < 160) 
        #         ]


plotting_parameters = pd.DataFrame(columns=["Station","Network","Latitude","Longitude","phi","dt","sks_count","s_deep_count","s_shallow_count","total_count"])

# Loop through stations
for station_code in station_list:
    time.sleep(1)
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    time.sleep(1)
    if STATION_PHASE_MAP[station_code] == 'complex':
        print("Station is complex and will not be analyzed.")
        continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]

        station_events = station_events[
            (station_events['evphase'] != 'SKS') |  # Keep if not SKS
            (station_events['calcbaz'] < 130) |      # OR if calcbaz < 145
            (station_events['calcbaz'] > 160)      # OR if calcbaz > 165
        ]

        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        
        result_binned = data['result_binned']
        station_events = data['station_events']
        
        # Extract parameters
        phi_binned = round(result_binned['phi'],1)
        dt_binned = round(result_binned['dt'],2)

        station_lat = round(station_events['evstationlat'].iloc[0],2)
        station_lon = round(station_events['evstationlon'].iloc[0],2)

        network_name = station_events['evnetwork'].iloc[0]
        print(network_name)

        sks_count = len(station_events[station_events['evphase'] == 'SKS'])
        s_deep_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] >= s_depth)])
        s_shallow_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] < s_depth)])
        total_count = sks_count + s_deep_count + s_shallow_count
        print(sks_count,s_deep_count,s_shallow_count)




        # print(phi_binned,dt_binned,station_lat,station_lon)

        plotting_parameters.loc[len(plotting_parameters)] = [station_code,network_name,station_lat,station_lon,phi_binned,dt_binned,sks_count,s_deep_count,s_shallow_count,total_count]
        # ######################################################
        # print(data['station_events'].head())


        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/nicest_plots/'

        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=True)
        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)

        # plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_plot.png', dpi=300, bbox_inches='tight')
        
        
        print(f"  ✓ Completed {station_code}")

        plt.close('all')

        results = []
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

# print(f"\n{'='*70}")
# print(f"✓ Processing complete for {len(station_list)} stations")
# print('='*70)

print(plotting_parameters)

plotting_parameters.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/publication_map/swp_measurements_no_cluster_jul292026.txt",sep='\t')

# make a table for the paper

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Load measurements
# ============================================================

new_measurements_path = "/work/gcl3/BR/nicolasv03/senior_thesis/measurements/publication_map/swp_measurements_corrected_jul292026.txt"

new_data_df = pd.read_csv(
    new_measurements_path,
    sep="\t",
    encoding="cp1252"
)

# ============================================================
# Build master station dataframe
# ============================================================

phase_df = pd.DataFrame(
    STATION_PHASE_MAP.items(),
    columns=["Station", "phase_dataset"]
)

station_df = phase_df.merge(
    new_data_df,
    on="Station",
    how="left"
)

# ============================================================
# Station classification
# ============================================================

station_df["classification"] = station_df["phase_dataset"].map({
    "sks": "Simple",
    "sks_deep": "Simple",
    "all": "Simple",
    "complex": "Complex"
}).fillna("Unknown")

# ============================================================
# Keep only phases actually used in final measurement
# ============================================================

station_df.loc[
    station_df.phase_dataset == "sks",
    ["s_deep_count", "s_shallow_count"]
] = 0

station_df.loc[
    station_df.phase_dataset == "sks_deep",
    "s_shallow_count"
] = 0

station_df["total_count"] = (
    station_df["sks_count"]
    + station_df["s_deep_count"]
    + station_df["s_shallow_count"]
)

# ============================================================
# Publication Table 1
# (Only interpreted stations)
# ============================================================

results_table = (
    station_df.loc[
        station_df["classification"] == "Simple",
        ["Station", "phase_dataset", "phi", "dt"]
    ]
    .copy()
)

results_table = results_table.rename(columns={
    "phase_dataset": "Phases",
    "phi": r"$\phi$",
    "dt": r"$\delta t$"
})

# nicer phase labels
results_table["Phases"] = results_table["Phases"].replace({
    "sks": "SKS",
    "sks_deep": "SKS + Deep S",
    "all": "SKS + Deep S + Shallow S"
})

# remove trailing zeros
for col in [r"$\phi$", r"$\delta t$"]:
    results_table[col] = results_table[col].map(lambda x: f"{x:g}")

# ============================================================
# Supplementary Table
# (Every processed station)
# ============================================================

catalog_table = station_df[
    [
        "Station",
        "classification",
        "phase_dataset",
        "total_count",
        "sks_count",
        "s_deep_count",
        "s_shallow_count",
    ]
].copy()

catalog_table = catalog_table.rename(columns={
    "classification": "Class",
    "phase_dataset": "Phases",
    "total_count": "Total",
    "sks_count": "SKS",
    "s_deep_count": "Deep S",
    "s_shallow_count": "Shallow S",
})

catalog_table["Phases"] = catalog_table["Phases"].replace({
    "sks": "SKS",
    "sks_deep": "SKS + Deep S",
    "all": "All",
    "complex": "-"
})

catalog_table = catalog_table.fillna("-")

for col in ["Total", "SKS", "Deep S", "Shallow S"]:
    catalog_table[col] = catalog_table[col].map(
        lambda x: f"{x:g}" if isinstance(x, (int, float, np.floating)) else x
    )

# ============================================================
# Export
# ============================================================

print(results_table.to_latex(index=False, escape=False))

print(catalog_table.to_latex(index=False, escape=False))

# AS31 double layer plot

In [5]:
'''
Doing this just to get splitting parameters for the stations so we can start making a publiation quality map. Final measurements coming soon!! May 11 2026

'''

# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Filter to only stations in dictionary
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
print(f"After dict filter: {len(station_list)} stations remaining")

print(station_list)

AS31_LIST = ["AS31"]

# Filter to only stations in dictionary
station_list = [s for s in station_list if s in AS31_LIST]
print(f"After dict filter: {len(station_list)} stations remaining")

# plotting_parameters = pd.DataFrame(columns=["Station","Network","Latitude","Longitude","phi","dt","sks_count","s_deep_count","s_shallow_count","total_count"])

# Loop through stations
for station_code in station_list:
    time.sleep(1)
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    time.sleep(1)
    # if STATION_PHASE_MAP[station_code] == 'complex':
    #     print("Station is complex and will not be analyzed.")
    #     continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]

        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        
        result_binned = data['result_binned']
        station_events = data['station_events']
        
        # # Extract parameters
        # phi_binned = round(result_binned['phi'],1)
        # dt_binned = round(result_binned['dt'],2)

        # station_lat = round(station_events['evstationlat'].iloc[0],2)
        # station_lon = round(station_events['evstationlon'].iloc[0],2)

        # network_name = station_events['evnetwork'].iloc[0]
        # print(network_name)

        # sks_count = len(station_events[station_events['evphase'] == 'SKS'])
        # s_deep_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] >= s_depth)])
        # s_shallow_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] < s_depth)])
        # total_count = sks_count + s_deep_count + s_shallow_count
        # print(sks_count,s_deep_count,s_shallow_count)




        # print(phi_binned,dt_binned,station_lat,station_lon)

        # plotting_parameters.loc[len(plotting_parameters)] = [station_code,network_name,station_lat,station_lon,phi_binned,dt_binned,sks_count,s_deep_count,s_shallow_count,total_count]
        # ######################################################
        # print(data['station_events'].head())


        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/as31_double_layer_plots/'

        # # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=True)
        fig7, ax7 = analysis.publication_SI_curve_for_AS31(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)

        plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_double_plot_aug28.png', dpi=300, bbox_inches='tight')
        
        
        print(f"  ✓ Completed {station_code}")

        plt.close('all')

        results = []
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

# print(f"\n{'='*70}")
# print(f"✓ Processing complete for {len(station_list)} stations")
# print('='*70)

# plotting_parameters.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/publication_map/swp_measurements_aug1_all_stations.txt",sep='\t')

Found 83 unique stations
Stations: ['ARMA', 'AS31', 'BAG', 'BBOO', 'BCON', 'BKB', 'BKNI', 'BLDU', 'BTDF', 'CAN', 'CISI', 'CMSA', 'CNB', 'COEN', 'CTA', 'CTAO', 'DAV', 'DLV', 'EIDS', 'FAKI', 'FITZ', 'FORT', 'GIRL', 'GUMO', 'HNR', 'HTT', 'INKA', 'IPM', 'JAGI', 'JAY', 'KDU', 'KKM', 'KLGGI', 'KMBL', 'KNRA', 'KOM', 'KSM', 'KUM', 'LCRK', 'LDM', 'LHI', 'LUWI', 'MANU', 'MEEK', 'MILA', 'MMRI', 'MNAI', 'MOO', 'MORW', 'MTN', 'MULG', 'MUN', 'NFK', 'NOUC', 'NWAO', 'OOD', 'PATS', 'PBKT', 'PMBI', 'PMG', 'QIS', 'QLP', 'RABL', 'RIV', 'SAUI', 'SBM', 'SKLT', 'SMRI', 'SNZO', 'SOEI', 'STKA', 'SZP', 'TNTI', 'TOLI', 'TOO', 'TROP', 'UBIN', 'UGM', 'WB2', 'WRKA', 'XMI', 'XMIS', 'YNG']

Processing 83 stations with ≥10 measurements
After dict filter: 59 stations remaining
['ARMA', 'AS31', 'BBOO', 'BKB', 'BKNI', 'BLDU', 'CAN', 'CMSA', 'CNB', 'COEN', 'CTA', 'CTAO', 'EIDS', 'FAKI', 'FITZ', 'FORT', 'GIRL', 'GUMO', 'HTT', 'INKA', 'JAGI', 'KDU', 'KMBL', 'KNRA', 'LCRK', 'LHI', 'LUWI', 'MANU', 'MEEK', 'MMRI', 'MNAI', 'MOO